In [2]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [3]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [4]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

Running on local — storage at: /home/luigi/RecSys
Running on local — storage at: /home/luigi/RecSys


<module 'Challenge.paths' from '/home/luigi/RecSys/Challenge/paths.py'>

In [5]:
# Load datasets
folds = paths.load_cv_folds(k=5)

## **Score blending**

In [6]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Load Models**

In [7]:
models = {}

### **SLIM**

In [8]:
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

# Train SLIM model with best parameters
# {"l1_ratio": 0.6257826858334088, "alpha": 0.000818987106299432, "topK": 646, "positive_only": false}

models['SLIM'] = []
for idx, (URM_train, URM_validation) in enumerate(folds):
    print(f"--- FOLD {idx+1} ---")

    # Train SLIM model
    slim_model = SLIMElasticNetRecommender(URM_train)
    slim_model.fit(
        l1_ratio=0.6257826858334088,
        alpha=0.000818987106299432,
        topK=646,
        positive_only=False
    )

    # Add SLIM model to models dictionary
    models['SLIM'].append(slim_model)

    # # Evaluate SLIM model
    # slim_recall = evaluate_recommender(slim_model, at=20, URM_validation=URM_validation)
    # print(f"SLIM Model - Recall@20: {slim_recall}")

--- FOLD 1 ---
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 3.62 min. Items per second: 32.10
--- FOLD 2 ---
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 3.33 min. Items per second: 34.88
--- FOLD 3 ---
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 3.76 min. Items per second: 30.88
--- FOLD 4 ---
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 3.63 min. Items per second: 31.96
--- FOLD 5 ---
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 3.49 min. Items per second: 33.31


### **EASE_R**

In [9]:
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender

# Train EASE_R model with best parameters
# {"topK": 850, "l2": 400.12671834453766, "normalize": false}

models['EASE_R'] = []
for idx, (URM_train, URM_validation) in enumerate(folds):
    print(f"--- FOLD {idx+1} ---")

    ease_r_model = EASE_R_Recommender(URM_train)
    ease_r_model.fit(
        topK=850,
        l2_norm=400.12671834453766,
        normalize_matrix=False
    )

    # Add EASE_R model to models dictionary
    models['EASE_R'].append(ease_r_model)

    # # Evaluate EASE_R model
    # ease_r_recall = evaluate_recommender(ease_r_model, at=20)
    # print(f"EASE_R Model - Recall@20: {ease_r_recall:.5f}")

--- FOLD 1 ---
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 7.09 sec
--- FOLD 2 ---
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 8.13 sec
--- FOLD 3 ---
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 8.48 sec
--- FOLD 4 ---
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 8.48 sec
--- FOLD 5 ---
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 8.77 sec


### **TopPop**

In [10]:
from Recommenders.NonPersonalizedRecommender import TopPop

models['TopPop'] = []

for idx, (URM_train, URM_validation) in enumerate(folds):
    # Train TopPop model
    toppop_model = TopPop(URM_train)
    toppop_model.fit()

    # Add TopPop model to models dictionary
    models['TopPop'].append(toppop_model)

    # # Evaluate TopPop model
    # toppop_recall = evaluate_recommender(toppop_model, at=20)
    # print(f"TopPop Model - Recall@20: {toppop_recall:.5f}")

### **KNN**

In [11]:
SIMILIARITIES = ["cosine", "pearson", "jaccard", "tversky"]

#### **UserKNN**

In [12]:
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender

params = {
    "cosine": {"topK": 347, "shrink": 3, "normalize": True, "feature_weighting": "TF-IDF"},
    "pearson": {"topK": 2850, "shrink": 935, "normalize": True, "feature_weighting": "TF-IDF"},
    "jaccard": {"topK": 301, "shrink": 0, "normalize": True, "feature_weighting": "none"},
    "tversky": {"topK": 401, "shrink": 5, "normalize": False, "feature_weighting": "TF-IDF", "alpha": 0.2184775930703794, "beta": 0.5934354684918872}
}

for sim in SIMILIARITIES:    
    
    models['UserKNN'+sim] = []
    for idx, (URM_train, URM_validation) in enumerate(folds):
        print(f"--- FOLD {idx+1} ---")
    
        # Train UserKNN model with best parameters
        userknn_model = UserKNNCFRecommender(URM_train)

        if sim == "tversky":
            userknn_model.fit(
                similarity=sim,
                topK=params[sim]["topK"],
                shrink=params[sim]["shrink"],
                normalize=params[sim]["normalize"],
                feature_weighting=params[sim]["feature_weighting"],
                tversky_alpha=params[sim]["alpha"],
                tversky_beta=params[sim]["beta"]
            )
        else:
            userknn_model.fit(
                similarity=sim,
                topK=params[sim]["topK"],
                shrink=params[sim]["shrink"],
                normalize=params[sim]["normalize"],
                feature_weighting=params[sim]["feature_weighting"]
            )

        # Add UserKNN model to models dictionary
        models['UserKNN'+sim].append(userknn_model)

        # # Evaluate UserKNN model
        # userknn_recall = evaluate_recommender(userknn_model, at=20)
        # print(f"UserKNN {sim} Model - Recall@20: {userknn_recall:.5f}")

--- FOLD 1 ---
Similarity column 27095 (100.0%), 1769.92 column/sec. Elapsed time 15.31 sec
--- FOLD 2 ---
Similarity column 27095 (100.0%), 1775.93 column/sec. Elapsed time 15.26 sec
--- FOLD 3 ---
Similarity column 27095 (100.0%), 1788.17 column/sec. Elapsed time 15.15 sec
--- FOLD 4 ---
Similarity column 27095 (100.0%), 1739.71 column/sec. Elapsed time 15.57 sec
--- FOLD 5 ---
Similarity column 27095 (100.0%), 1800.41 column/sec. Elapsed time 15.05 sec
--- FOLD 1 ---
Similarity column 27095 (100.0%), 1868.82 column/sec. Elapsed time 14.50 sec
--- FOLD 2 ---
Similarity column 27095 (100.0%), 1836.49 column/sec. Elapsed time 14.75 sec
--- FOLD 3 ---
Similarity column 27095 (100.0%), 1870.39 column/sec. Elapsed time 14.49 sec
--- FOLD 4 ---
Similarity column 27095 (100.0%), 1818.01 column/sec. Elapsed time 14.90 sec
--- FOLD 5 ---
Similarity column 27095 (100.0%), 1865.27 column/sec. Elapsed time 14.53 sec
--- FOLD 1 ---
Similarity column 27095 (100.0%), 1853.24 column/sec. Elapsed tim

#### **ItemKNN**

In [13]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

params = {
    "cosine": {"topK": 79, "shrink": 431, "normalize": True, "feature_weighting": "TF-IDF"},
    "pearson": {"topK": 11, "shrink": 3, "normalize": True, "feature_weighting": "TF-IDF"},
    "jaccard": {"topK": 1101, "shrink": 0, "normalize": True, "feature_weighting": "BM25"},
    "tversky": {"topK": 6, "shrink": 106, "normalize": True, "feature_weighting": "TF-IDF", "tversky_alpha": 0.10252983277640502, "tversky_beta": 0.9684276681445425}
}

for sim in SIMILIARITIES:

    models['ItemKNN'+sim] = []
    for idx, (URM_train, URM_validation) in enumerate(folds):
        print(f"--- FOLD {idx+1} ---")

        # Train ItemKNN model with best parameters
        itemknn_model = ItemKNNCFRecommender(URM_train)
        
        if sim == "tversky":
            itemknn_model.fit(
                similarity=sim,
                topK=params[sim]["topK"],
                shrink=params[sim]["shrink"],
                normalize=params[sim]["normalize"],
                feature_weighting=params[sim]["feature_weighting"],
                tversky_alpha=params[sim]["tversky_alpha"],
                tversky_beta=params[sim]["tversky_beta"]
            )
        else:
            itemknn_model.fit(
                similarity=sim,
                topK=params[sim]["topK"],
                shrink=params[sim]["shrink"],
                normalize=params[sim]["normalize"],
                feature_weighting=params[sim]["feature_weighting"]
            )

        # Add ItemKNN model to models dictionary
        models['ItemKNN'+sim].append(itemknn_model)

        # # Evaluate ItemKNN model
        # itemknn_recall = evaluate_recommender(itemknn_model, at=20)
        # print(f"ItemKNN {sim} Model - Recall@20: {itemknn_recall:.5f}")

--- FOLD 1 ---
Similarity column 6969 (100.0%), 4813.09 column/sec. Elapsed time 1.45 sec
--- FOLD 2 ---
Similarity column 6969 (100.0%), 4666.35 column/sec. Elapsed time 1.49 sec
--- FOLD 3 ---
Similarity column 6969 (100.0%), 4717.57 column/sec. Elapsed time 1.48 sec
--- FOLD 4 ---
Similarity column 6969 (100.0%), 4520.74 column/sec. Elapsed time 1.54 sec
--- FOLD 5 ---
Similarity column 6969 (100.0%), 4761.64 column/sec. Elapsed time 1.46 sec
--- FOLD 1 ---
Similarity column 6969 (100.0%), 4813.85 column/sec. Elapsed time 1.45 sec
--- FOLD 2 ---
Similarity column 6969 (100.0%), 4722.00 column/sec. Elapsed time 1.48 sec
--- FOLD 3 ---
Similarity column 6969 (100.0%), 4776.38 column/sec. Elapsed time 1.46 sec
--- FOLD 4 ---
Similarity column 6969 (100.0%), 4824.68 column/sec. Elapsed time 1.44 sec
--- FOLD 5 ---
Similarity column 6969 (100.0%), 4721.00 column/sec. Elapsed time 1.48 sec
--- FOLD 1 ---
Similarity column 6969 (100.0%), 4640.44 column/sec. Elapsed time 1.50 sec
--- FOLD 2

## **Best score blending**

In [14]:
models

{'SLIM': [<Recommenders.SLIM.SLIMElasticNetRecommender.SLIMElasticNetRecommender at 0x7f10dbbb6660>,
 'EASE_R': [<Recommenders.EASE_R.EASE_R_Recommender.EASE_R_Recommender at 0x7f10dbbb6cf0>,
 'TopPop': [<Recommenders.NonPersonalizedRecommender.TopPop at 0x7f10dbbb6f90>,
 'UserKNNcosine': [<Recommenders.KNN.UserKNNCFRecommender.UserKNNCFRecommender at 0x7f10dbbb70e0>,
 'UserKNNpearson': [<Recommenders.KNN.UserKNNCFRecommender.UserKNNCFRecommender at 0x7f1156d7e210>,
 'UserKNNjaccard': [<Recommenders.KNN.UserKNNCFRecommender.UserKNNCFRecommender at 0x7f10db5f5310>,
 'UserKNNtversky': [<Recommenders.KNN.UserKNNCFRecommender.UserKNNCFRecommender at 0x7f10db6769c0>,
 'ItemKNNcosine': [<Recommenders.KNN.ItemKNNCFRecommender.ItemKNNCFRecommender at 0x7f10dbbb7620>,
 'ItemKNNpearson': [<Recommenders.KNN.ItemKNNCFRecommender.ItemKNNCFRecommender at 0x7f10dbbcfd10>,
 'ItemKNNjaccard': [<Recommenders.KNN.ItemKNNCFRecommender.ItemKNNCFRecommender at 0x7f11543e4140>,
 'ItemKNNtversky': [<Recommend

In [15]:
len(models)

11

In [ ]:
# save all the models
for key in models:
    for idx, model in enumerate(models[key]):
        if 'KNN' not in key:
            model.save_model(os.path.join(paths.MODEL_DIR, "hybrid", f"{key}_fold{idx+1}"))

SLIMElasticNetRecommender: Saving model in file '/home/luigi/RecSys/models/hybrid/SLIM_fold1SLIMElasticNetRecommender'
SLIMElasticNetRecommender: Saving complete
SLIMElasticNetRecommender: Saving model in file '/home/luigi/RecSys/models/hybrid/SLIM_fold2SLIMElasticNetRecommender'
SLIMElasticNetRecommender: Saving complete
SLIMElasticNetRecommender: Saving model in file '/home/luigi/RecSys/models/hybrid/SLIM_fold3SLIMElasticNetRecommender'
SLIMElasticNetRecommender: Saving complete
SLIMElasticNetRecommender: Saving model in file '/home/luigi/RecSys/models/hybrid/SLIM_fold4SLIMElasticNetRecommender'
SLIMElasticNetRecommender: Saving complete
SLIMElasticNetRecommender: Saving model in file '/home/luigi/RecSys/models/hybrid/SLIM_fold5SLIMElasticNetRecommender'
SLIMElasticNetRecommender: Saving complete
EASE_R_Recommender: Saving model in file '/home/luigi/RecSys/models/hybrid/EASE_R_fold1EASE_R_Recommender'
EASE_R_Recommender: Saving complete
EASE_R_Recommender: Saving model in file '/home

KeyboardInterrupt: 

In [ ]:
for rec in models.keys():
    print(f"Recommender: {rec}")
    
    validation_scores = []
    for idx, (URM_train, URM_validation) in enumerate(folds):        
        recommender = models[rec][idx]
        
        # Evaluate
        score = evaluate_recommender(recommender, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {len(validation_scores)} - Score: {score}")

    print(f"Average validation score: {np.mean(validation_scores)}\n")

In [40]:
def get_scores(recommender, user_id, cutoff=20):
    items, scores = recommender.recommend(user_id, cutoff=cutoff, return_scores=True)
    return items, scores[0]

In [43]:
# Precompute all scores for each model and each user
URM_train, _ = folds[0]
user_ids = range(URM_train.shape[0])


cached_scores = {}
for idx in range(5):
    cached_scores[idx] = {}
    
    for user_id in user_ids:
        if user_id % 1000 == 0:
            print(f"Processing user {user_id} in fold {idx}")

        recommended_items = {}
        for model_key in models.keys():
            recommender = models[model_key][idx]
            items, scores = get_scores(recommender, user_id, cutoff=40)
            
            for item, score in zip(items, scores):
                if item not in recommended_items:
                    recommended_items[item] = {}
                recommended_items[item][model_key] = score

        items = list(recommended_items.keys())
        scores_matrix = None
        for model_key in models.keys():
            scores_array = np.array([recommended_items[item].get(model_key, 0.0) for item in items])
            
            # Stack scores as columns
            if scores_matrix is None:
                scores_matrix = scores_array.reshape(-1, 1)
            else:
                scores_matrix = np.hstack((scores_matrix, scores_array.reshape(-1, 1)))

        cached_scores[idx][user_id] = {
            'recommended_items': items,
            'scores_matrix': scores_matrix
        }

Processing user 0 in fold 0
Processing user 1000 in fold 0
Processing user 2000 in fold 0
Processing user 3000 in fold 0
Processing user 4000 in fold 0
Processing user 5000 in fold 0
Processing user 6000 in fold 0
Processing user 7000 in fold 0
Processing user 8000 in fold 0
Processing user 9000 in fold 0
Processing user 10000 in fold 0
Processing user 11000 in fold 0
Processing user 12000 in fold 0
Processing user 13000 in fold 0
Processing user 14000 in fold 0
Processing user 15000 in fold 0
Processing user 16000 in fold 0
Processing user 17000 in fold 0
Processing user 18000 in fold 0
Processing user 19000 in fold 0
Processing user 20000 in fold 0
Processing user 21000 in fold 0
Processing user 22000 in fold 0
Processing user 23000 in fold 0
Processing user 24000 in fold 0
Processing user 25000 in fold 0
Processing user 26000 in fold 0
Processing user 27000 in fold 0
Processing user 0 in fold 1
Processing user 1000 in fold 1
Processing user 2000 in fold 1
Processing user 3000 in fol

In [44]:
# Save to disk
import pickle

filepath = os.path.join(paths.MODEL_DIR, "hybrid", "cached_scores.pkl")
with open(filepath, 'wb') as f:
    pickle.dump(cached_scores, f)

## **Blend scores**

In [24]:
class ScoreBlendingRecommender:
    def __init__(self, models, weights, cutoff_multiplier=2):
        self.models = models
        self.weights = weights
        self.cutoff_multiplier = cutoff_multiplier
    
    def recommend(self, user_id, cutoff=20):
        adjusted_cutoff = cutoff * self.cutoff_multiplier
        recommendations = {}
        
        for model, w in zip(self.models, self.weights):
            items, scores = model.recommend(user_id,cutoff=adjusted_cutoff, return_scores=True)
            scores = scores[0] # Extract scores from nested structure
            
            for item, score in zip(items, scores):
                if item not in recommendations:
                    recommendations[item] = 0.0
                recommendations[item] += score * w

        # Sort items by aggregated score
        sorted_items = sorted(recommendations.items(), key=lambda x: x[1], reverse=True)
        recommended_items = [item for item, score in sorted_items[:cutoff]]
        
        return recommended_items

In [37]:
from Challenge.hyper_tuning import ModelOptimizer
import optuna
optimizer = ModelOptimizer("ScoreBlendingRecommender")

STUDY_NAME = "ScoreBlendingRecommender4"

In [38]:
recommnders_keys = list(models.keys())

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    weights = [
        optuna_trial.suggest_float(f"weight_{i}", 0.0, 0.3)
        for i in range(10)
    ]
    weights.insert(0, 1.0) # Max to SLIM

    # Normalization isn't strictly necessary here
    # Doesn't change the relative importance of models
    # and so the final ranking
    # Print normalized because are more interpretable weights
    w_to_model = {recommnders_keys[i]: w / sum(weights) for i,w in enumerate(weights)}
    # sort by weights
    w_to_model = dict(sorted(w_to_model.items(), key=lambda item: item[1], reverse=True))
    # pretty print
    print("Current weights:")
    for key, w in w_to_model.items():
        print(f"  {key}: {w:.4f}")
    print()
    
    validation_scores = []
    for idx, (_, URM_validation) in enumerate(folds):        
        # Train the recommender
        recommender_instance = ScoreBlendingRecommender(
            models=[ models[key][idx] for key in recommnders_keys ],
            weights=weights
        )
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)

        # Show fold result
        print(f"Fold {idx+1} - Score: {score}")

        curr_avg = np.mean(validation_scores)
        # if curr_avg < 0.28:
        #     raise optuna.TrialPruned()

        # Report intermediate result to Optuna
        optuna_trial.report(curr_avg, idx+1)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            raise optuna.TrialPruned()
        
        # Log fold performance
        optimizer.log_fold_performance(idx+1, score)

    avg_score = np.mean(validation_scores)
    print(f"Average validation score: {avg_score:.4f}")

    return avg_score

In [39]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

[I 2025-11-10 01:05:26,834] A new study created in RDB with name: ScoreBlendingRecommender4


  0%|          | 0/100 [00:00<?, ?it/s]

Current weights:
  SLIM: 0.3725
  UserKNNtversky: 0.1044
  ItemKNNjaccard: 0.0991
  UserKNNcosine: 0.0917
  UserKNNjaccard: 0.0895
  UserKNNpearson: 0.0606
  ItemKNNtversky: 0.0549
  ItemKNNpearson: 0.0540
  TopPop: 0.0363
  EASE_R: 0.0241
  ItemKNNcosine: 0.0129

Fold 1 - Score: 0.08879997581243515
Fold 2 - Score: 0.08949774503707886
Fold 3 - Score: 0.08835332095623016
Fold 4 - Score: 0.08824625611305237
Fold 5 - Score: 0.0885404422879219
Average validation score: 0.0887
[I 2025-11-10 01:11:43,880] Trial 0 finished with value: 0.08868754655122757 and parameters: {'weight_0': 0.06468839501219824, 'weight_1': 0.09737245637290565, 'weight_2': 0.24620895048880584, 'weight_3': 0.16254453928679133, 'weight_4': 0.24037028073132372, 'weight_5': 0.2801314034255575, 'weight_6': 0.03468105750563687, 'weight_7': 0.14491363882746627, 'weight_8': 0.26606652502730593, 'weight_9': 0.14748775489820018}. Best is trial 0 with value: 0.08868754655122757.
Current weights:
  SLIM: 0.4175
  UserKNNjaccard: 

KeyboardInterrupt: 

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

### **Retrain models on train+validation set**

In [ ]:
print("###   Retrain models on train+validation set   ###")

URM_train, URM_validation = folds[0]  # Using the first fold just to get the datasets

print("Training SLIM model...")
slim_model = SLIMElasticNetRecommender(URM_train+URM_validation)
slim_model.fit(
    l1_ratio=0.6257826858334088,
    alpha=0.000818987106299432,
    topK=646,
    positive_only=False
)
models['SLIM'] = slim_model

print("Training EASE_R model...")
ease_r_model = EASE_R_Recommender(URM_train+URM_validation)
ease_r_model.fit(
    topK=850,
    l2_norm=400.12671834453766,
    normalize_matrix=False
)
models['EASE_R'] = ease_r_model

print("Training TopPop model...")
# Train TopPop model
toppop_model = TopPop(URM_train+URM_validation)
toppop_model.fit()
models['TopPop'] = toppop_model

print("Training UserKNN models...")
params = {
    "cosine": {"topK": 347, "shrink": 3, "normalize": True, "feature_weighting": "TF-IDF"},
    "pearson": {"topK": 2850, "shrink": 935, "normalize": True, "feature_weighting": "TF-IDF"},
    "jaccard": {"topK": 301, "shrink": 0, "normalize": True, "feature_weighting": "none"},
    "tversky": {"topK": 401, "shrink": 5, "normalize": False, "feature_weighting": "TF-IDF", "alpha": 0.2184775930703794, "beta": 0.5934354684918872}
}

for sim in SIMILIARITIES:
    # Train UserKNN model with best parameters
    print(f"  {sim}...")
    userknn_model = UserKNNCFRecommender(URM_train+URM_validation)

    if sim == "tversky":
        userknn_model.fit(
            similarity=sim,
            topK=params[sim]["topK"],
            shrink=params[sim]["shrink"],
            normalize=params[sim]["normalize"],
            feature_weighting=params[sim]["feature_weighting"],
            tversky_alpha=params[sim]["alpha"],
            tversky_beta=params[sim]["beta"]
        )
    else:
        userknn_model.fit(
            similarity=sim,
            topK=params[sim]["topK"],
            shrink=params[sim]["shrink"],
            normalize=params[sim]["normalize"],
            feature_weighting=params[sim]["feature_weighting"]
        )

    # Add UserKNN model to models dictionary
    models['UserKNN'+sim] = userknn_model


print("Training ItemKNN models...")
params = {
    "cosine": {"topK": 79, "shrink": 431, "normalize": True, "feature_weighting": "TF-IDF"},
    "pearson": {"topK": 11, "shrink": 3, "normalize": True, "feature_weighting": "TF-IDF"},
    "jaccard": {"topK": 1101, "shrink": 0, "normalize": True, "feature_weighting": "BM25"},
    "tversky": {"topK": 6, "shrink": 106, "normalize": True, "feature_weighting": "TF-IDF", "tversky_alpha": 0.10252983277640502, "tversky_beta": 0.9684276681445425}
}

for sim in SIMILIARITIES:
    print(f"  {sim}...")
    itemknn_model = ItemKNNCFRecommender(URM_train+URM_validation)
    
    if sim == "tversky":
        itemknn_model.fit(
            similarity=sim,
            topK=params[sim]["topK"],
            shrink=params[sim]["shrink"],
            normalize=params[sim]["normalize"],
            feature_weighting=params[sim]["feature_weighting"],
            tversky_alpha=params[sim]["tversky_alpha"],
            tversky_beta=params[sim]["tversky_beta"]
        )
    else:
        itemknn_model.fit(
            similarity=sim,
            topK=params[sim]["topK"],
            shrink=params[sim]["shrink"],
            normalize=params[sim]["normalize"],
            feature_weighting=params[sim]["feature_weighting"]
        )

    # Add ItemKNN model to models dictionary
    models['ItemKNN'+sim] = itemknn_model

### **Generate submissions**

In [ ]:
import pandas as pd

# Initialize the final recommender with the best weights
best_weights = []
for i in range(11):
    best_weights.append(optuna_study.best_trial.params[f"weight_{i}"])

recommender = ScoreBlendingRecommender(
    models=[ models[key] for key in recommnders_keys ],
    weights=best_weights
)

# Generate recommendations for the test set
user_ids_test = pd.read_csv(paths.CHALLENGE_USER_IDS_TEST)
ids = user_ids_test["user_id"].values

os.makedirs(paths.SUBMISSIONS, exist_ok=True)
with open(os.path.join(paths.SUBMISSIONS, "hybridModelToUser.csv"), "w") as f:
    f.write("user_id,item_list\n")
    for user_id in ids:
        recommendations = recommender.recommend(user_id, cutoff=20)
        f.write(f"{user_id},{' '.join([str(item) for item in recommendations])}\n")